<!-- # Large Language Models -->

# Post-Training, Chatbots and Agents


<!-- ::: {.callout-tip}
# Acknowledgement
The following tutorial is a summary of Andrej Karpathy's video [A Deep Dive into Large Language Models like ChatGPT](https://www.youtube.com/watch?v=Te5rOTcE3FQ). I highly recommend watching the video, as it contains a lot of useful information and visualizations that are hard to capture in text.

Note: If you are looking for the excalidraw diagram that Andrej made for the video, you can download it [here](https://cdn.anfalmushtaq.com/static/imgs/blog/2025-02-08-deep-dive-into-llms-like-chatgpt-tldr/LLMIntro-2024-07-22-1743.excalidraw). He shared it through Google Drive and it invalidates the link after a certain time. That's why I have decided to host it on my CDN as well.
:::  -->

Everything we have covered in the course so far is generally referred to as Pre-Training. In fact, the P in GPT stands for Pre-training. 

Despite the fact that the pre-training phase is quite intensive, it does not necessarily produce a model that is useful for real-world applications. The model needs to be adapted to specific tasks or objectives, which is where Post-Training comes in.

| Pillar	| Foundational publication | 
|:---:|:---:|
| Architecture	| Attention Is All You Need |
| Pre-training	| GPT-1, GPT-2 |
| Scaling	| Scaling Laws, GPT-3 |
| Post-training	| InstructGPT, ChatGPT |

There are a lot of terms thrown around in the context of large language models (LLMs) like ChatGPT. In this tutorial, we will try to clarify some of these terms and concepts. 

Both GPT-1 and GPT-2 were accompanied by research papers published by OpenAI.


There is a particularly interesting historical arc here:

- GPT-1: Pre-train a model and fine-tune it for each task.
- GPT-2: Pre-train a sufficiently large model and attempt tasks without fine-tuning.
- GPT-3: Teach the model new tasks through examples in the prompt.
- InstructGPT: Fine-tune the model to follow instructions generally, rather than for one specific task.
- ChatGPT: Adapt the model into a general-purpose conversational assistant.

In some ways, InstructGPT represents a return to the pre-training and fine-tuning paradigm of GPT-1, but with a fundamentally different objective for fine-tuning.


## 0. Pre-Attention History 

| Year | Concept | Publication / Resource | Main Contributors |
|---|---|---|---|
| 1805 | Least Squares | [Nouvelles méthodes pour la détermination des orbites des comètes](https://archive.org/search?query=Legendre+Nouvelles+methodes+1805) | [Adrien-Marie Legendre](https://en.wikipedia.org/wiki/Adrien-Marie_Legendre) |
| 1809 | Least Squares | [Theoria Motus Corporum Coelestium](https://archive.org/search?query=Gauss+Theoria+Motus+1809) | [Carl Friedrich Gauss](https://en.wikipedia.org/wiki/Carl_Friedrich_Gauss) |
| 1847 | Gradient Descent | [Méthode générale pour la résolution des systèmes d'équations simultanées](https://scholar.google.com/scholar?q=Cauchy+1847+Methode+generale+resolution) | [Augustin-Louis Cauchy](https://en.wikipedia.org/wiki/Augustin-Louis_Cauchy) |
| 1886 | Linear Regression | [Regression Towards Mediocrity in Hereditary Stature](https://doi.org/10.2307/2841583) | [Francis Galton](https://en.wikipedia.org/wiki/Francis_Galton) |
| 1943 | Artificial Neuron | [A Logical Calculus of the Ideas Immanent in Nervous Activity](https://doi.org/10.1007/BF02478259) | [Warren McCulloch](https://en.wikipedia.org/wiki/Warren_McCulloch), [Walter Pitts](https://en.wikipedia.org/wiki/Walter_Pitts) |
| 1948 | Entropy and Information Theory | [A Mathematical Theory of Communication](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf) | [Claude Shannon](https://en.wikipedia.org/wiki/Claude_Shannon) |
| 1951 | KL Divergence and Cross-Entropy Foundations | [On Information and Sufficiency](https://doi.org/10.1214/aoms/1177729694) | [Solomon Kullback](https://en.wikipedia.org/wiki/Solomon_Kullback), [Richard Leibler](https://en.wikipedia.org/wiki/Richard_Leibler) |
| 1951 | Stochastic Gradient Descent Foundations | [A Stochastic Approximation Method](https://doi.org/10.1214/aoms/1177729586) | [Herbert Robbins](https://en.wikipedia.org/wiki/Herbert_Robbins), [Sutton Monro](https://en.wikipedia.org/w/index.php?search=Sutton+Monro) |
| 1958 | Perceptron | [The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain](https://doi.org/10.1037/h0042519) | [Frank Rosenblatt](https://en.wikipedia.org/wiki/Frank_Rosenblatt) |
| 1962 | Multi-Layer Perceptron Foundations | [Principles of Neurodynamics](https://archive.org/search?query=Rosenblatt+Principles+of+Neurodynamics) | [Frank Rosenblatt](https://en.wikipedia.org/wiki/Frank_Rosenblatt) |
| 1970 | Reverse-Mode Automatic Differentiation | [The Representation of the Cumulative Rounding Error of an Algorithm as a Taylor Expansion](https://doi.org/10.1007/BF01931367) (1976 English publication) | [Seppo Linnainmaa](https://en.wikipedia.org/wiki/Seppo_Linnainmaa) |
| 1974 | Backpropagation Foundations | [Beyond Regression: New Tools for Prediction and Analysis in the Behavioral Sciences](https://scholar.google.com/scholar?q=Paul+Werbos+Beyond+Regression+1974) | [Paul Werbos](https://en.wikipedia.org/wiki/Paul_Werbos) |
| 1986 | Backpropagation | [Learning Representations by Back-Propagating Errors](https://doi.org/10.1038/323533a0) | [David Rumelhart](https://en.wikipedia.org/wiki/David_Rumelhart), [Geoffrey Hinton](https://en.wikipedia.org/wiki/Geoffrey_Hinton), [Ronald Williams](https://en.wikipedia.org/wiki/Ronald_J._Williams) |
| 1989 | Universal Approximation Theorem | [Approximation by Superpositions of a Sigmoidal Function](https://doi.org/10.1007/BF02551274) | [George Cybenko](https://en.wikipedia.org/wiki/George_Cybenko) |
| 1998 | Transfer Learning | [Learning to Learn](https://doi.org/10.1007/978-1-4615-5529-2) | [Sebastian Thrun](https://en.wikipedia.org/wiki/Sebastian_Thrun), [Lorien Pratt](https://en.wikipedia.org/wiki/Lorien_Pratt) |
| 1998 | Convolutional Neural Networks (LeNet) | [Gradient-Based Learning Applied to Document Recognition](https://doi.org/10.1109/5.726791) | [Yann LeCun](https://en.wikipedia.org/wiki/Yann_LeCun), [Léon Bottou](https://en.wikipedia.org/wiki/L%C3%A9on_Bottou), [Yoshua Bengio](https://en.wikipedia.org/wiki/Yoshua_Bengio), [Patrick Haffner](https://en.wikipedia.org/w/index.php?search=Patrick+Haffner) |
| 2007 | Scikit-learn (Initial Development) | [Scikit-learn: Project History](https://scikit-learn.org/stable/about.html) | [David Cournapeau](https://en.wikipedia.org/w/index.php?search=David+Cournapeau) |
| 2010 | Transfer Learning Survey | [A Survey on Transfer Learning](https://doi.org/10.1109/TKDE.2009.191) | [Sinno Jialin Pan](https://en.wikipedia.org/w/index.php?search=Sinno+Jialin+Pan), [Qiang Yang](https://en.wikipedia.org/wiki/Qiang_Yang_(computer_scientist)) |
| 2010 | ReLU | [Rectified Linear Units Improve Restricted Boltzmann Machines](https://www.cs.toronto.edu/~hinton/absps/reluICML.pdf) | [Vinod Nair](https://en.wikipedia.org/w/index.php?search=Vinod+Nair+computer+scientist), [Geoffrey Hinton](https://en.wikipedia.org/wiki/Geoffrey_Hinton) |
| 2011 | ReLU in Deep Feedforward Networks | [Deep Sparse Rectifier Neural Networks](https://proceedings.mlr.press/v15/glorot11a.html) | [Xavier Glorot](https://en.wikipedia.org/w/index.php?search=Xavier+Glorot), [Antoine Bordes](https://en.wikipedia.org/w/index.php?search=Antoine+Bordes), [Yoshua Bengio](https://en.wikipedia.org/wiki/Yoshua_Bengio) |
| 2011 | Scikit-learn | [Scikit-learn: Machine Learning in Python](https://jmlr.org/papers/v12/pedregosa11a.html) | [Fabian Pedregosa](https://en.wikipedia.org/w/index.php?search=Fabian+Pedregosa) et al. |
| 2012 | Dropout | [Improving Neural Networks by Preventing Co-adaptation of Feature Detectors](https://arxiv.org/abs/1207.0580) | [Geoffrey Hinton](https://en.wikipedia.org/wiki/Geoffrey_Hinton) et al. |
| 2012 | AlexNet and GPU-Accelerated Deep Learning | [ImageNet Classification with Deep Convolutional Neural Networks](https://proceedings.neurips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks) | [Alex Krizhevsky](https://en.wikipedia.org/wiki/Alex_Krizhevsky), [Ilya Sutskever](https://en.wikipedia.org/wiki/Ilya_Sutskever), [Geoffrey Hinton](https://en.wikipedia.org/wiki/Geoffrey_Hinton) |
| 2014 | Dropout | [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html) | [Nitish Srivastava](https://en.wikipedia.org/w/index.php?search=Nitish+Srivastava) et al. |
| 2014 | Transfer Learning in Deep Neural Networks | [How Transferable Are Features in Deep Neural Networks?](https://arxiv.org/abs/1411.1792) | [Jason Yosinski](https://en.wikipedia.org/w/index.php?search=Jason+Yosinski) et al. |
| 2014 | Adam Optimizer | [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980) | [Diederik P. Kingma](https://en.wikipedia.org/wiki/Diederik_P._Kingma), [Jimmy Ba](https://en.wikipedia.org/wiki/Jimmy_Ba) |
| 2014 | Neural Attention | [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473) | [Dzmitry Bahdanau](https://en.wikipedia.org/w/index.php?search=Dzmitry+Bahdanau), [Kyunghyun Cho](https://en.wikipedia.org/wiki/Kyunghyun_Cho), [Yoshua Bengio](https://en.wikipedia.org/wiki/Yoshua_Bengio) |
| 2015 | Batch Normalization | [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167) | [Sergey Ioffe](https://en.wikipedia.org/w/index.php?search=Sergey+Ioffe), [Christian Szegedy](https://en.wikipedia.org/wiki/Christian_Szegedy) |
| 2015 | Residual Connections | [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385) | [Kaiming He](https://en.wikipedia.org/wiki/Kaiming_He) et al. |
| 2016 | Layer Normalization | [Layer Normalization](https://arxiv.org/abs/1607.06450) | [Jimmy Ba](https://en.wikipedia.org/wiki/Jimmy_Ba), [Jamie Kiros](https://en.wikipedia.org/w/index.php?search=Jamie+Ryan+Kiros), [Geoffrey Hinton](https://en.wikipedia.org/wiki/Geoffrey_Hinton) |
| 2017 | PyTorch (Initial Release) | [PyTorch GitHub Repository](https://github.com/pytorch/pytorch) | [Facebook AI Research](https://en.wikipedia.org/wiki/Meta_AI), [Adam Paszke](https://en.wikipedia.org/w/index.php?search=Adam+Paszke), [Soumith Chintala](https://en.wikipedia.org/wiki/Soumith_Chintala) |
| 2017 | Transformer | [Attention Is All You Need](https://arxiv.org/abs/1706.03762) | [Ashish Vaswani](https://en.wikipedia.org/wiki/Ashish_Vaswani), [Noam Shazeer](https://en.wikipedia.org/wiki/Noam_Shazeer), [Niki Parmar](https://en.wikipedia.org/wiki/Niki_Parmar), [Jakob Uszkoreit](https://en.wikipedia.org/wiki/Jakob_Uszkoreit), [Llion Jones](https://en.wikipedia.org/wiki/Llion_Jones), [Aidan Gomez](https://en.wikipedia.org/wiki/Aidan_Gomez), [Łukasz Kaiser](https://en.wikipedia.org/wiki/%C5%81ukasz_Kaiser), [Illia Polosukhin](https://en.wikipedia.org/wiki/Illia_Polosukhin) |
| 2017 | AdamW Optimizer | [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101) | [Ilya Loshchilov](https://en.wikipedia.org/w/index.php?search=Ilya+Loshchilov), [Frank Hutter](https://en.wikipedia.org/wiki/Frank_Hutter) |
| 2018 | Generative Pre-Training (GPT-1) | [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) | [Alec Radford](https://en.wikipedia.org/wiki/Alec_Radford), [Karthik Narasimhan](https://en.wikipedia.org/w/index.php?search=Karthik+Narasimhan), [Tim Salimans](https://en.wikipedia.org/w/index.php?search=Tim+Salimans), [Ilya Sutskever](https://en.wikipedia.org/wiki/Ilya_Sutskever) |
| 2019 | PyTorch | [PyTorch: An Imperative Style, High-Performance Deep Learning Library](https://arxiv.org/abs/1912.01703) | [Adam Paszke](https://en.wikipedia.org/w/index.php?search=Adam+Paszke), [Sam Gross](https://en.wikipedia.org/w/index.php?search=Sam+Gross+PyTorch), [Soumith Chintala](https://en.wikipedia.org/wiki/Soumith_Chintala) et al. |

## 1. Transformer (2017)

<img src="https://arxiv.org/html/1706.03762v7/Figures/ModalNet-21.png" width="40%" align="right" style="filter: invert(1);">

_[Attention Is All You Need](https://arxiv.org/abs/1706.03762)_ Vaswani et al. — June 2017, published at NeurIPS 2017. Authors from Google Brain and Google Research.

Key contributions:

- Introduced the transformer architecture.

- Replaced recurrence and convolution with attention mechanisms.

- Introduced multi-head attention and positional encodings.

- Used an encoder-decoder architecture for machine translation.

- Enabled more parallelizable training than recurrent neural networks.


## 2. GPT-1 (2018)

<img src="gpt1.png" width="25%" align="right" style="filter: invert(1);">

_[Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf?utm_source=chatgpt.com)_ Radford, Narasimhan, Salimans, and Sutskever — June 2018. Authors from OpenAI.

Key contributions:

- Introduced the Generative Pre-trained Transformer (GPT).

- Used a decoder-only transformer with 117 million parameters, trained on the BooksCorpus dataset (7,000 unpublished books). Layers = 12, Hidden size = 768, Attention heads = 12.

- Combined self-supervised pre-training with <mark>task-specific supervised fine-tuning</mark>.

- Demonstrated the <mark>effectiveness of transfer learning</mark> for natural language processing.



## 3. GPT-2 (2019)

_[Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf?utm_source=chatgpt.com)_ Radford et al. — February 2019. Authors from OpenAI.

Key contributions:

- Scaled GPT to 1.5 billion parameters (12 layers, 48 attention heads).

- Introduced the WebText training dataset.

- Demonstrated zero-shot learning across multiple NLP tasks.

Zero-shot learning refers to the ability of a model to perform a task without having seen any examples of that task during training. In the context of GPT-2, this means that the model can generate text or answer questions about topics it has never been explicitly trained on, simply by leveraging its understanding of language and context learned during pre-training.

- Showed that language models could perform certain downstream tasks without task-specific fine-tuning.



## 4. Scaling Laws (2020)

_[Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361)_, Kaplan et al. — January 2020.

Key contributions:


- Demonstrated that language model performance improves predictably with scale.

- Showed that architectural details such as depth and width have relatively little impact within a broad range, compared to overall scale.

- Investigated optimal allocation of computational resources during training.

- Provided a scientific foundation for training increasingly large language models

- Established empirical power-law relationships between language modeling loss and three fundamental quantities:

  1. Model size (number of parameters).

  2. Training dataset size (number of tokens).
  
  3. Computational budget (FLOPs).

<img src="scaling_laws1.png" width="100%" style="filter: invert(1)">

In the Kaplan et al. (2020) paper, [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361?utm_source=chatgpt.com), the plots use three main quantities:

| Symbol  | Meaning                                  |
| ------- | ---------------------------------------- |
| \\(N\\) | Number of non-embedding model parameters |
| \\(D\\) | Dataset size in tokens                   |
| \\(C\\) | Compute budget in FLOPs                  |
| \\(L\\) | Cross-entropy loss                       |

The legends often distinguish different model sizes or compute budgets. Each curve represents training under a particular configuration.

The straight lines on log-log plots indicate a power-law relationship between loss and the quantity being scaled.

For example:

$$ L(N)=\left(\frac{N_c}{N}\right)^{\alpha_N} $$

Here, $N_c$ is a fitted scaling constant and $\alpha_N$ is the scaling exponent, which determines how quickly loss decreases as model size increases.

If you mean a specific figure's legend, send me the plot and I can explain each entry precisely.

The plots above show the relationship between Test Loss and Compute (left panel), Dataset Size (center panel) and Model Size (right panel). The plots are log-log plots, which means that the axes are logarithmic. This is important because it means that the relationship between the variables is not linear, but rather follows a power law. 

Generally:
$$
\text{Compute} \approx 6 \times N \times D \times E
$$

where \(N\) is parameter count, \(D\) is dataset size in tokens, and \(E\) is the number of epochs.

<img src="scaling_laws2.png" width="45%" style="filter: invert(1)" align="right">

Compute here refers to the total number of floating point operations (FLOPs) used to train the model. Dataset Size refers to the number of tokens in the training dataset, and Model Size refers to the number of parameters in the model. The PF-days on the x-axis of the left panel refers to petaFLOP-days, which is a measure of computational cost. One petaFLOP-day is equal to 10^15 floating point operations per day.


The plot on the right shows sample efficiency curves for different model sizes. The x-axis is the number of training tokens, and the y-axis is the test loss. Each curve corresponds to a different model size, and we can see that <mark>larger models tend to perform better (lower test loss) than smaller models, given the same amount of training data</mark>.

### FLOPs


Floating Point Operations (FLOPs) are a measure of computational cost. They count the number of arithmetic operations (additions, multiplications, etc.) that a model performs during training or inference. FLOPs give us a hardware-independent way to answer questions like *"How expensive is it to train this model?"* or *"How much compute does one response cost?"*

FLOPs is easy to confuse with FLOPS

* **FLOPs** (lowercase *s*) → a *quantity*: the total number of operations. Example: "Training this model took $10^{23}$ FLOPs."
* **FLOPS** or **FLOP/s** → a *rate*: operations per second, a property of hardware. Example: "This GPU delivers about $10^{15}$ FLOP/s."

Dividing the first by the second gives you time: $\text{time} = \text{FLOPs} / \text{FLOP/s}$.

Almost all the compute in a Transformer is matrix multiplication. Multiplying an $m \times k$ matrix by a $k \times n$ matrix produces $m \times n$ outputs, and each output is a dot product of length $k$ ($k$ multiplications and roughly $k$ additions). So:

$$
\text{FLOPs}(\text{matmul}) \approx 2mnk
$$

**Example:** Passing a batch of $B$ token vectors of size $d_{in}$ through a linear layer with weight matrix $d_{in} \times d_{out}$ costs $2 \cdot B \cdot d_{in} \cdot d_{out}$ FLOPs.

Notice that $d_{in} \cdot d_{out}$ is just the number of parameters in that layer. This leads to a very useful rule of thumb.

For a model with $N$ parameters:

* **Forward pass:** each parameter is used in about one multiply and one add per token, so
  $$\text{FLOPs}_{\text{forward}} \approx 2N \text{ per token}$$
* **Backward pass:** we compute gradients with respect to both the activations and the weights, which costs about twice the forward pass:
  $$\text{FLOPs}_{\text{backward}} \approx 4N \text{ per token}$$

Adding them together gives the most important formula in this section:

$$
\boxed{C_{\text{train}} \approx 6ND}
$$

where $C$ is the total training compute in FLOPs, $N$ is the number of parameters, and $D$ is the number of training tokens.

For **inference**, there is no backward pass, so generating each token costs roughly $2N$ FLOPs.

> **Note:** This approximation ignores the attention score computation ($QK^\top$ and the weighted sum over $V$), which grows with context length. For typical context lengths it is a small fraction of the total, but for very long contexts it becomes significant.

How much compute does it take to train a 7-billion-parameter model on 2 trillion tokens?

$$
C \approx 6 \times (7 \times 10^9) \times (2 \times 10^{12}) = 8.4 \times 10^{22} \text{ FLOPs}
$$

**Converting to time:** An NVIDIA H100 GPU has a peak of roughly $10^{15}$ FLOP/s for dense BF16 math. In practice, real training runs reach only about 30–50% of peak. This fraction is called **Model FLOPs Utilization (MFU)**, and it is lower because of memory movement, communication between GPUs, and other overhead. Assuming 40% MFU:

$$
\text{effective throughput} \approx 0.4 \times 10^{15} = 4 \times 10^{14} \text{ FLOP/s}
$$

$$
\text{GPU-seconds} \approx \frac{8.4 \times 10^{22}}{4 \times 10^{14}} \approx 2.1 \times 10^{8} \text{ s} \approx 58{,}000 \text{ GPU-hours}
$$

On a cluster of 1,000 GPUs, that's about **2.5 days** of training.

FLOPs are the currency of scaling laws. Given a fixed compute budget $C$, we have to decide how to split it between a bigger model (larger $N$) and more data (larger $D$), since $C \approx 6ND$.



## 5. GPT-3 (2020)

[Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165?utm_source=chatgpt.com)

Brown et al. — May 2020, subsequently published at NeurIPS 2020.

Key contributions:

- Scaled GPT to 175 billion parameters, layer depth of 96, and context length of 2,048 tokens.

- Demonstrated in-context learning.

- Established that models could learn new tasks from examples provided in the prompt without updating model parameters.


|                       | GPT-1             | GPT-2              | GPT-3               |
| --------------------- | ----------------- | ------------------ | ------------------- |
| Year                  | 2018              | 2019               | 2020                |
| Parameters            | 117M              | 1.5B               | 175B                |
| Transformer layers    | 12                | 48                 | 96                  |
| Context length        | 512               | 1,024              | 2,048               |
| Defining contribution | Transfer learning | Zero-shot learning | Few-shot learning   |
| Task adaptation       | Fine-tuning       | Prompting          | In-context learning |

<img src="gpt3.png" width="100%" style="filter: invert(1)">


## 6. Chinchilla (2022)

_[Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556?utm_source=chatgpt.com)_ Hoffmann et al. — March 2022, published at ICLR 2022. Authors from DeepMind.

Challenged the conclusions of Kaplan et al., 2020, which suggested that larger models should be trained on more data. Instead, <mark>Chinchilla found that smaller models trained on more data could outperform larger models trained on less data, given the same compute budget</mark>. 

Compute budget refers to the total number of floating point operations (FLOPs) used to train the model. The Chinchilla paper claims that the optimal model size and dataset size should be balanced, with approximately <mark>20 tokens per parameter</mark>.

The **Chinchilla** study (Hoffmann et al., 2022) found that compute-optimal training uses roughly **20 tokens per parameter**:

$$
D_{\text{optimal}} \approx 20N
$$

For example, Chinchilla itself was a 70B-parameter model trained on 1.4T tokens. Many modern models deliberately train on far *more* tokens than this. That costs extra compute during training but produces a smaller model that is cheaper to run at inference time, where costs are paid every time a user sends a request.

<img src="chinchilla.png" width="100%" style="filter: invert(1)">

Chinchilla (Hoffmann et al., 2022) is a follow-up to Kaplan et al., 2020, which showed that language-model loss falls as a smooth power law in model size, data, and compute, and <mark>concluded that most of any extra compute should go to making models bigger rather than training them on more data</mark>. Chinchilla revisited this with more careful experiments (notably, adjusting the learning-rate schedule to each run's length) and found that parameters and training tokens should grow in roughly equal proportion, about 20 tokens per parameter. <mark>To show this, they trained Chinchilla (70B parameters) with the same compute as their own 280B-parameter Gopher model, and Chinchilla outperformed it, which meant many earlier large models had been undertrained</mark>.





## 7. InstructGPT (2022)

_[Training Language Models to Follow Instructions with Human Feedback](https://arxiv.org/pdf/2203.02155)_ Ouyang et al. — March 2022, published at NeurIPS 2022.

Key contributions:

- Fine-tuned GPT-3 to follow natural language instructions.

- Established a three-stage post-training pipeline:

  - Supervised fine-tuning (SFT).

  - Reward modeling (RM).

  - Reinforcement learning using PPO.

- Demonstrated that a 1.3-billion-parameter aligned model was preferred by human evaluators over the 175-billion-parameter GPT-3 base model.

- Established the importance of aligning language models with human intent rather than simply increasing model size. 

<img src="instructgpt.png" width="100%" style="filter: invert(1)">



## 8. ChatGPT (2022)

_[Introducing ChatGPT](https://openai.com/index/chatgpt/?utm_source=chatgpt.com)_ OpenAI — November 30, 2022.

Key contributions:

- Adapted GPT-3.5 into a general-purpose conversational assistant.

- Extended the InstructGPT training methodology to multi-turn dialogue.

- Incorporated human-generated conversations into supervised fine-tuning.

- Used reinforcement learning from human feedback (RLHF) to improve conversational behavior.

- Made large language models accessible to the general public through a conversational interface.

## 9. GPT-4 (2023)



## 10. o1 and Reasoning Models (2024)

OpenAI o1 (September 2024): a new scaling axis. Until then, “scaling” meant bigger models, more data and more training compute. 

<mark>o1 showed a second axis, test-time compute: letting the model think longer before answering improves accuracy smoothly and predictably, the same way training compute does</mark>. 

Its gains on competition math and coding were large. OpenAI published few details about how it was trained.

## 11. o3 and ARC-AGI (2025)

Test-time scaling on unfamiliar problems. 

ARC-AGI is a puzzle benchmark designed to resist memorization, and earlier LLMs did very poorly on it. 

o3 scored dramatically higher, but only by spending a very large amount of compute per task. 

That made it both an impressive demonstration and a cautionary example of how expensive test-time scaling can get.

---



## Keeping Track of LLMs

If you're interested in following developments in this space, here are some great resources:

* [Arena](https://lmarena.ai/) (formerly LMArena / Chatbot Arena) → Crowdsourced, blind head-to-head rankings of language models.
* [Artificial Analysis](https://artificialanalysis.ai/) → Benchmark scores, speed, and pricing across models.
* [AI News](https://news.smol.ai/) → A daily newsletter summarizing AI research and discussion.

## Where to Find LLMs

Want to try different LLMs? Here's where to find them:

* **Proprietary Models** → OpenAI (GPT), Google (Gemini), Anthropic (Claude), etc.
* **Open-Weight Models** → DeepSeek, Qwen, Kimi, GLM, Gemma, gpt-oss, Mistral, etc. Try them via [Together.ai](https://www.together.ai/) or [OpenRouter](https://openrouter.ai/).
* **Run Locally** → Use [Ollama](https://ollama.com/) or [LM Studio](https://lmstudio.ai/).
* **Base Models** → Download base checkpoints (e.g., Qwen or OLMo base models) from [Hugging Face](https://huggingface.co/models) and compare them with their instruct versions.